## Data Cleaning

In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/healthcare_appointments_raw.csv")

In [2]:
cleaned_df = df.copy()

In [ ]:
# Check duplicates

cleaned_df.duplicated().sum()

np.int64(0)

In [ ]:
# Check duplicate Appointment IDs

cleaned_df["AppointmentID"].duplicated().sum()

np.int64(0)

### Check invalid ages

In [5]:
cleaned_df["Age"].describe()

count    110527.000000
mean         37.088874
std          23.110205
min          -1.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: Age, dtype: float64

In [6]:
cleaned_df[cleaned_df["Age"]<0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


In [7]:
cleaned_df["Age"].min()

np.int64(-1)

In [8]:
cleaned_df["Age"].max()

np.int64(115)

### Checking categorical values

In [9]:
cleaned_df["Gender"].value_counts()

Gender
F    71840
M    38687
Name: count, dtype: int64

In [10]:
cleaned_df["Scholarship"].value_counts()

Scholarship
0    99666
1    10861
Name: count, dtype: int64

In [11]:
cleaned_df["Hipertension"].value_counts()

Hipertension
0    88726
1    21801
Name: count, dtype: int64

In [12]:
cleaned_df["Diabetes"].value_counts()

Diabetes
0    102584
1      7943
Name: count, dtype: int64

In [13]:
cleaned_df["Alcoholism"].value_counts()

Alcoholism
0    107167
1      3360
Name: count, dtype: int64

In [21]:
cleaned_df["Handcap"].value_counts()

Handcap
0    108286
1      2042
2       183
3        13
4         3
Name: count, dtype: int64

In [14]:
cleaned_df["SMS_received"].value_counts()

SMS_received
0    75045
1    35482
Name: count, dtype: int64

### Patients with Invalid Ages


In [15]:
cleaned_df[cleaned_df["Age"] == -1].shape[0]

1

In [16]:
cleaned_df[cleaned_df["Age"] == -1]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


In [17]:
cleaned_df[cleaned_df["Age"] == 115]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
63912,3.196321e+13,5700278,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
63915,3.196321e+13,5700279,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
68127,3.196321e+13,5562812,F,2016-04-08T14:29:17Z,2016-05-16T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
76284,3.196321e+13,5744037,F,2016-05-30T09:44:51Z,2016-05-30T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,No
97666,7.482346e+14,5717451,F,2016-05-19T07:57:56Z,2016-06-03T00:00:00Z,115,SÃO JOSÉ,0,1,0,0,0,1,No


In [18]:
cleaned_df[cleaned_df["PatientId"] == 99832]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show


### Fixing the invalid age

In [19]:
cleaned_df.loc[cleaned_df["Age"] == -1, "Age"] = cleaned_df["Age"].median()

In [20]:
cleaned_df["Age"].min()

np.int64(0)

In [22]:
cleaned_df[cleaned_df["Age"] == -1].shape[0]

0

In [23]:
cleaned_df[cleaned_df["PatientId"] == 99832]["Age"]

Series([], Name: Age, dtype: int64)

## Rename the columns

In [24]:
cleaned_df = cleaned_df.rename(columns={
    "Hipertension": "Hypertension" ,
    "Handcap": "Handicap",
    "No-show": "No_show"
    })

In [25]:
cleaned_df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension',
       'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'No_show'],
      dtype='str')

## Date Conversion

In [26]:
cleaned_df["ScheduledDay"] = pd.to_datetime(cleaned_df["ScheduledDay"])
cleaned_df["AppointmentDay"] = pd.to_datetime(cleaned_df["AppointmentDay"])

In [28]:
cleaned_df[["ScheduledDay", "AppointmentDay"]].dtypes

ScheduledDay      datetime64[us, UTC]
AppointmentDay    datetime64[us, UTC]
dtype: object

In [29]:
(cleaned_df["AppointmentDay"] < cleaned_df["ScheduledDay"]).sum()

np.int64(38568)

## Compare calendar dates

In [30]:
scheduled_date = cleaned_df["ScheduledDay"].dt.date
appointment_date = cleaned_df["AppointmentDay"].dt.date

In [31]:
(scheduled_date > appointment_date).sum()

np.int64(5)

## Find and Remove the 5 invalid records

In [32]:
cleaned_df[scheduled_date > appointment_date]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_show
27033,7.839273e+12,5679978,M,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,38,RESISTÊNCIA,0,0,0,0,1,0,Yes
55226,7.896294e+12,5715660,F,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,19,SANTO ANTÔNIO,0,0,0,0,1,0,Yes
64175,2.425226e+13,5664962,F,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,22,CONSOLAÇÃO,0,0,0,0,0,0,Yes
71533,9.982316e+14,5686628,F,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,81,SANTO ANTÔNIO,0,0,0,0,0,0,Yes
72362,3.787482e+12,5655637,M,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,7,TABUAZEIRO,0,0,0,0,0,0,Yes


In [33]:
invalid_dates = cleaned_df["ScheduledDay"].dt.date > cleaned_df["AppointmentDay"].dt.date

cleaned_df = cleaned_df[~invalid_dates].copy()

In [34]:
print("Rows after removing invalid dates:", cleaned_df.shape[0])

Rows after removing invalid dates: 110522


In [35]:
print(
    "Remaining invalid date records:",
    (
        cleaned_df["ScheduledDay"].dt.date
        > cleaned_df["AppointmentDay"].dt.date
    ).sum()
)

Remaining invalid date records: 0


## Create WaitingDays

In [36]:
cleaned_df["WaitingDays"] = (
    cleaned_df["AppointmentDay"].dt.normalize()
    - cleaned_df["ScheduledDay"].dt.normalize()
).dt.days

In [37]:
cleaned_df["WaitingDays"].describe()

count    110522.000000
mean         10.184253
std          15.255115
min           0.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

In [38]:
cleaned_df["WaitingDays"].min()

np.int64(0)

In [39]:
cleaned_df["WaitingDays"].max()

np.int64(179)

## Check the target variable

In [40]:
cleaned_df["No_show"].value_counts()

No_show
No     88208
Yes    22314
Name: count, dtype: int64

In [41]:
cleaned_df["No_show"].value_counts(normalize=True) * 100

No_show
No     79.810354
Yes    20.189646
Name: proportion, dtype: float64

## Final cleaning check

In [42]:
cleaned_df.info()

<class 'pandas.DataFrame'>
Index: 110522 entries, 0 to 110526
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   PatientId       110522 non-null  float64            
 1   AppointmentID   110522 non-null  int64              
 2   Gender          110522 non-null  str                
 3   ScheduledDay    110522 non-null  datetime64[us, UTC]
 4   AppointmentDay  110522 non-null  datetime64[us, UTC]
 5   Age             110522 non-null  int64              
 6   Neighbourhood   110522 non-null  str                
 7   Scholarship     110522 non-null  int64              
 8   Hypertension    110522 non-null  int64              
 9   Diabetes        110522 non-null  int64              
 10  Alcoholism      110522 non-null  int64              
 11  Handicap        110522 non-null  int64              
 12  SMS_received    110522 non-null  int64              
 13  No_show         110522 non-nul

In [43]:
cleaned_df.isnull().sum()

PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hypertension      0
Diabetes          0
Alcoholism        0
Handicap          0
SMS_received      0
No_show           0
WaitingDays       0
dtype: int64

In [44]:
cleaned_df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_show,WaitingDays
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,No,0
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,No,0
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,No,0
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,0
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,No,0


## Saving the cleaned dataset

In [46]:
cleaned_df.to_csv(
    "../data/cleaned/healthcare_appointments_cleaned.csv",
    index=False
)

In [47]:
import os

os.path.exists("../data/cleaned/healthcare_appointments_cleaned.csv")

True